# Ingestion Pipeline

End-to-end pipeline that:
1. Loads `.txt` knowledge-base files
2. Parses metadata from file headers
3. Chunks the documents
4. Embeds the chunks
5. Stores them in the vector database
6. Runs a sanity-check retrieval query

**Reuses** logic from `1_load_chunk.ipynb`, `2_embeddings.ipynb`, `3_retrieval.ipynb`, `4_basic_RAG.ipynb` and `src/` py files.

## Setup 
### Imports & Paths

In [11]:
!pip install -r ../requirements.txt

     -------------------------------------- 177.1/177.1 kB 3.5 MB/s eta 0:00:00
     ---------------------------------------- 2.1/2.1 MB 26.5 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from pathlib import Path
import sys
import os
import numpy as np
from dotenv import load_dotenv

# ── Project root & src on path ──────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# ── Knowledge-base directories ───────────────────────────────────────────────
LGBT_EU_BY_COUNTRY_DIR = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_country"
LGBT_EU_BY_SUBSET_DIR  = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_subset"
HIV_KB_DIR             = PROJECT_ROOT / "data" / "3_txt_KB" / "HIV_AIDS_data"
UNICEF_KB_DIR          = PROJECT_ROOT / "data" / "3_txt_KB" / "UNICEF_Immunization"

KB_DIRS = [
    LGBT_EU_BY_COUNTRY_DIR,
    LGBT_EU_BY_SUBSET_DIR,
    HIV_KB_DIR,
    UNICEF_KB_DIR,
]

print("Project root:", PROJECT_ROOT)
for d in KB_DIRS:
    status = "✓" if d.exists() else "✗ (not found)"
    print(f"  {status}  {d.relative_to(PROJECT_ROOT)}")

Project root: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG
  ✓  data\3_txt_KB\LGBT_EU\by_country
  ✓  data\3_txt_KB\LGBT_EU\by_subset
  ✓  data\3_txt_KB\HIV_AIDS_data
  ✓  data\3_txt_KB\UNICEF_Immunization


In [13]:
# ── Standard library & third-party ───────────────────────────────────────────
from typing import Dict, List, Tuple

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# ── src modules (reused from previous notebooks) ─────────────────────────────
from vectorstore import build_vectorstore
from retrieval   import retrieve, print_results, format_context
from llm         import build_prompt, ask


In [14]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

API key loaded: ✅


In [15]:
# used in evaluation stage
import nltk
import re
import numpy as np
import nltk
from sklearn.metrics.pairwise import cosine_similarity
from textstat import flesch_reading_ease

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from pathlib import Path

## Parameters

- uses `RecursiveCharacterTextSplitter` used in `1_load_chunk.ipynb`.
- Edit params to change how it will be applied.

In [16]:
# whether we actually use OpenAI credits on this
USE_OPENAI = True

# ── Chunking parameters (mirrors 1_load_chunk.ipynb) ─────────────────────────
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

# ── Vector-store persistence path ─────────────────────────────────────────────
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore"

# ── Embedding model (mirrors 2_embeddings.ipynb) ──────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"

# ── Retrieval parameters (mirrors 3_retrieval.ipynb) ─────────────────────────
TOP_K = 18

print(f"CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}")
print(f"EMBEDDING_MODEL={EMBEDDING_MODEL}")
print(f"VECTORSTORE_DIR={VECTORSTORE_DIR}")

CHUNK_SIZE=500, CHUNK_OVERLAP=50
EMBEDDING_MODEL=text-embedding-3-small
VECTORSTORE_DIR=C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


## Load All `.txt` Files
- load in from knowledge base (the output of `7_generate_text_knowledge_base.ipynb`)

In [17]:
def collect_txt_files(directories: List[Path]) -> List[Path]:
    """Recursively collect every .txt file from the given directories."""
    files: List[Path] = []
    for directory in directories:
        if not directory.exists():
            print(f"  ⚠  Directory not found, skipping: {directory}")
            continue
        found = sorted(directory.rglob("*.txt"))
        print(f"  Found {len(found):>4} files in {directory.relative_to(PROJECT_ROOT)}")
        files.extend(found)
    return files


all_txt_files = collect_txt_files(KB_DIRS)
print(f"\nTotal .txt files: {len(all_txt_files)}")

  Found 4427 files in data\3_txt_KB\LGBT_EU\by_country
  Found  699 files in data\3_txt_KB\LGBT_EU\by_subset
  Found   69 files in data\3_txt_KB\HIV_AIDS_data
  Found  292 files in data\3_txt_KB\UNICEF_Immunization

Total .txt files: 5487


## Handle Metadata + Content

In [18]:
def parse_document(file_path: Path) -> Tuple[str, Dict[str, str]]:
    raw = file_path.read_text(encoding="utf-8")
    lines = raw.splitlines()

    metadata: Dict[str, str] = {}
    content_start = 0

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "":          # blank line → header ends here
            content_start = i + 1
            break

        if ":" in stripped:         # metadata line  KEY: value
            key, _, value = stripped.partition(":")
            metadata[key.strip().lower()] = value.strip()
        else:
            # Not a metadata line and not blank → no header, treat whole file as content
            content_start = 0
            metadata = {}
            break

    content = "\n".join(lines[content_start:]).strip()
    metadata["source"] = str(file_path)

    return content, metadata

In [19]:
# ── Quick smoke-test on the first available file ──────────────────────────────
if all_txt_files:
    _sample_content, _sample_meta = parse_document(all_txt_files[0])
    print("Sample file :", all_txt_files[0].name)
    print("Metadata    :", _sample_meta)
    print("Content (100 chars):", _sample_content[:100], "...")
else:
    print("No files found — check KB_DIRS above.")

Sample file : b1_a_answer_by_Austria.txt
Metadata    : {'dataset': 'EU_LGBT', 'question_code': 'b1_a', 'subset': 'Austria', 'source': 'C:\\Users\\RAZER\\Desktop\\portfolio-projects\\1. RAG\\data\\3_txt_KB\\LGBT_EU\\by_country\\LGBT_Survey_DailyLife\\b1_a_answer_by_Austria.txt'}
Content (100 chars): Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual a ...


In [20]:
def build_documents(file_paths: List[Path]) -> List[Document]:
    """
    Parse every file and return a list of LangChain Documents.
    Mirrors the Document creation pattern from 1_load_chunk.ipynb.
    """
    docs: List[Document] = []
    errors: List[str] = []

    for fp in file_paths:
        try:
            content, metadata = parse_document(fp)
            if content:            # skip empty files
                docs.append(Document(page_content=content, metadata=metadata))
        except Exception as exc:
            errors.append(f"{fp.name}: {exc}")

    if errors:
        print(f"⚠  {len(errors)} file(s) could not be parsed:")
        for e in errors:
            print("  ", e)

    print(f"\nDocuments created : {len(docs)}")
    return docs


raw_documents = build_documents(all_txt_files)


Documents created : 5487


## Chunk Documents

Reuses the `RecursiveCharacterTextSplitter` we made `1_load_chunk.ipynb`.

In [21]:
# ── Text splitter — mirrors 1_load_chunk.ipynb ────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True,   # keeps track of character offset (used in 1_load_chunk.ipynb)
)

chunks = text_splitter.split_documents(raw_documents)

print(f"Raw documents : {len(raw_documents)}")
print(f"Chunks        : {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // max(len(chunks), 1)} chars")

Raw documents : 5487
Chunks        : 30830
Avg chunk size: 352 chars


In [22]:
# ── Inspect a sample chunk ────────────────────────────────────────────────────
if chunks:
    sample = chunks[0]
    print("Sample chunk metadata :", sample.metadata)
    print("Sample chunk content  :", sample.page_content[:200], "...")

Sample chunk metadata : {'dataset': 'EU_LGBT', 'question_code': 'b1_a', 'subset': 'Austria', 'source': 'C:\\Users\\RAZER\\Desktop\\portfolio-projects\\1. RAG\\data\\3_txt_KB\\LGBT_EU\\by_country\\LGBT_Survey_DailyLife\\b1_a_answer_by_Austria.txt', 'start_index': 0}
Sample chunk content  : Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Austria responses (Bisexual ...


## Embed Documents
- Reuses the `OpenAIEmbeddings` setup from `2_embeddings.ipynb`.

In [23]:
# ── Embedding model — mirrors 2_embeddings.ipynb ──────────────────────────────
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Quick sanity check: embed a single string
_test_vec = embeddings.embed_query("test")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Vector dimension: {len(_test_vec)}")

Embedding model : text-embedding-3-small
Vector dimension: 1536


## Store in Vector DB
- Reuses `load_vectorstore` from `src/vectorstore.py`.

In [24]:
# ── Persist directory ─────────────────────────────────────────────────────────
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Vector-store directory: {VECTORSTORE_DIR}")

Vector-store directory: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


In [25]:
# ── Build / overwrite the vector store ───────────────────────────────────────
# load_vectorstore is expected to accept (chunks, embeddings, persist_directory)
# and return a Chroma (or equivalent) vectorstore — as used in 3_retrieval.ipynb

vectorstore = build_vectorstore(
    documents=chunks,
    embeddings=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

print(f"\n✓ Vector store built and persisted to: {VECTORSTORE_DIR}")
print(f"  Total vectors stored: {vectorstore._collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Built vectorstore: 92490 chunks

✓ Vector store built and persisted to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore
  Total vectors stored: 92490


C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\src\vectorstore.py:32: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


# Testing
## Small test of End-to-End Retrieval
- Runs a sample query through the full pipeline
- Same structure as `4_basic_RAG.ipynb`.

In [26]:
# SAMPLE_QUERY = "What is the HIV prevalence rate in Eastern Europe?"
# SAMPLE_QUERY = "Do Lesbians in Romania experience a better or worse daily life experience than Bisexual women in Romania?"
SAMPLE_QUERY = "Lesbian vs Bisexual women, daily life experience in Romania"
print(f"Sample query: {SAMPLE_QUERY}")

Sample query: Lesbian vs Bisexual women, daily life experience in Romania


### 1. Retrieve relevant chunks


In [27]:
# retrieve() mirrors 3_retrieval.ipynb usage
results = retrieve(
    query=SAMPLE_QUERY,
    vectorstore=vectorstore,
    k=TOP_K,
)

print(f"Retrieved {len(results)} chunks:\n")
print_results(query = SAMPLE_QUERY, results = results)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 18 chunks:

Query: 'Lesbian vs Bisexual women, daily life experience in Romania'

--- Result 1 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLife\b2_a_answer_by_Bisexualwomen.txt
Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disagree: 0%, Current situation is fine: 1%, Don`t know: 7%

--- Result 2 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLife\b2_a_answer_by_Bisexualwomen.txt
Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | 

### 2. Format context

In [28]:
context = format_context(results)
print("Context passed to LLM (first 500 chars):")
print(context[:500], "...")

Context passed to LLM (first 500 chars):
Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disagree: 0%, Current situation is fine: 1%, Don`t know: 7%

Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you l ...


### 3. Build prompt

In [29]:
# build_prompt() mirrors 4_basic_RAG.ipynb usage
prompt = build_prompt(query=SAMPLE_QUERY, context=context)
print("Prompt (first 500 chars):")
print(prompt[:500], "...")

Prompt (first 500 chars):
[SystemMessage(content="You are a helpful assistant. Answer the user's question using only the context provided below. If the answer is not in the context, say 'I don't have enough information to answer that.'", additional_kwargs={}, response_metadata={}), HumanMessage(content='Context:\nQuestion b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disagree: 0%, Current situation is fine: 1%, Don`t know: 7%\n\nQuestion b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disa

### 4. Ask the LLM

In [30]:
# ask() mirrors 4_basic_RAG.ipynb usage
answer = ask(prompt, context)
print("=" * 60)
print("QUESTION:", SAMPLE_QUERY)
print("=" * 60)
print("ANSWER:")
print(answer)

QUESTION: Lesbian vs Bisexual women, daily life experience in Romania
ANSWER:
I don't have enough information to answer that.


## Pipeline Summary

In [31]:
print("Pipeline complete ✓")
print(f"  Files loaded    : {len(all_txt_files)}")
print(f"  Documents parsed: {len(raw_documents)}")
print(f"  Chunks created  : {len(chunks)}")
print(f"  Vectors stored  : {vectorstore._collection.count()}")
print(f"  Vector store at : {VECTORSTORE_DIR}")

Pipeline complete ✓
  Files loaded    : 5487
  Documents parsed: 5487
  Chunks created  : 30830
  Vectors stored  : 92490
  Vector store at : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


## RAG Self-Evaluation

In [32]:
# Install lightweight evaluation dependencies (skip if already present)
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("textstat")
_ensure("openpyxl")
_ensure("nltk")

nltk.download("punkt",      quiet=True)
nltk.download("punkt_tab",  quiet=True)
print("Dependencies ready ✅")

Dependencies ready ✅


In [33]:
TEST_QUESTIONS = [
    # LGBT EU survey
    "What percentage of gay men in Germany experienced discrimination in the past year?",
    "How comfortable do lesbian women in France feel being open about their identity at work?",
    "What share of transgender people in Poland reported hate-motivated violence?",
    "Compare acceptance levels of same-sex couples in Sweden versus Hungary.",
    # HIV / AIDS
    "What is the HIV prevalence rate among adults in sub-Saharan Africa?",
    "How has antiretroviral therapy coverage changed over the past decade?",
    # UNICEF Immunization
    "What is the global vaccination coverage rate for measles in children under five?",
    "Which regions have the lowest DTP3 immunization rates according to UNICEF data?",
]
print(f"Running evaluation on {len(TEST_QUESTIONS)} questions...")

Running evaluation on 8 questions...


In [34]:
def _tokenize(text: str) -> set:
    """Lowercase word tokens, punctuation stripped."""
    return set(re.findall(r"\b[a-z]+\b", text.lower()))


def retrieval_metrics(query: str, chunks, embeddings_model) -> dict:
    """Compute retrieval-level metrics for a list of LangChain Documents."""
    texts = [c.page_content for c in chunks]

    # Duplicate ratio
    unique_texts = set(texts)
    dup_ratio = round(1 - len(unique_texts) / len(texts), 3) if texts else 0.0

    # Unique source files
    sources = [c.metadata.get("source", "") for c in chunks]
    unique_src = len(set(sources))

    # Cosine similarity between query embedding and chunk embeddings
    try:
        q_vec  = np.array(embeddings_model.embed_query(query)).reshape(1, -1)
        c_vecs = np.array(embeddings_model.embed_documents(list(unique_texts)))
        cos_scores = cosine_similarity(q_vec, c_vecs)[0]
        cos_avg = round(float(cos_scores.mean()), 4)
    except Exception:
        cos_avg = None

    return {
        "n_chunks_retrieved": len(texts),
        "unique_sources":     unique_src,
        "duplicate_ratio":    dup_ratio,
        "query_chunk_cosine_avg": cos_avg,
    }


def answer_metrics(query: str, answer: str, chunks) -> dict:
    """Compute answer-quality metrics."""
    no_info_phrases = [
        "don't have enough information",
        "do not have enough information",
        "cannot answer",
        "not in the context",
    ]
    answered = not any(p in answer.lower() for p in no_info_phrases)

    answer_words = _tokenize(answer)
    chunk_words  = set()
    for c in chunks:
        chunk_words |= _tokenize(c.page_content)
    query_words = _tokenize(query)

    groundedness   = round(len(answer_words & chunk_words) / len(answer_words), 3) if answer_words else 0.0
    relevancy      = round(len(answer_words & query_words) / len(query_words),  3) if query_words  else 0.0
    word_count     = len(answer.split())
    try:
        flesch = round(flesch_reading_ease(answer), 1)
    except Exception:
        flesch = None

    return {
        "answer_length_words":  word_count,
        "answered":             answered,
        "groundedness":         groundedness,
        "relevancy_score":      relevancy,
        "flesch_reading_ease":  flesch,
    }


print("Helper functions defined ✅")

Helper functions defined ✅
